This notebook prcesses raw skeletons by breaking branches, and remerging the fragments. After all strips from a given directory are processed, they are then combined using translation and offset information.

In [12]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import navis
import os
import requests
import glob
from cloudvolume import CloudVolume, Skeleton
import numpy as np
from pathlib import Path
from io import BytesIO
import concurrent.futures
from joblib import dump,load, Parallel, delayed, parallel_config
from natsort import natsorted
import uuid
from ac_segmentation.reconnect_stack_navis import reconnect, read_navis_neurons_tar, write_navis_skels_tar, swap_dimensions, apply_transform_skeletons, remove_translate_nodes, swc_split_branches

In [15]:
#Declare directories and model file paths
sc = load("/ACdata/Users/connorl/Models/scaler.joblib") #scalar file
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib") #model file

skel_dir = "/allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_processing_demo/data/skeleton_outputs/H17_x55_S32_230412_highres.zarr/"
out_dir = "/ACdata/Users/connorl/Skeletons/H17_x55_S32_230412_highres_MIP1/"
files = natsorted(glob.glob(skel_dir + "*.gz"))

os.makedirs(out_dir + "/Reconnected/", exist_ok = True)
os.makedirs(out_dir + "/Translated/", exist_ok = True) 

In [16]:
###Create bounding box and translation lists
bound_boxs = [[[0,22848,0,41.9,0,288],[0,22848,0,288,0,134]]]*78 + [[[0,22848,0,41.9,0,288]]]*39

xoff = np.zeros(len(files))
yoff = np.tile(np.array(range(39))*np.array(-246.1), 3)
zoff = np.repeat(np.array([0,-154,-308]), repeats=39)
translations = np.array((xoff,yoff,zoff)).T

In [ ]:
###Reconnect, translate, and filter individual strips
def postprocess_strip(out_dir, file, cl, sc, bound_boxs, min_nodes=20, trans=[0,0,0], query_dis=20, resample=4, smooth=2, min_collin=.7):
    #reconnect skeletons
    print(file)
    name = file.split("/")[-1]
    skels, merges = reconnect(skels=file, cl=cl, sc=sc, query_dis=query_dis, min_collin=min_collin, resample=resample, smooth=smooth)
    skels = navis.NeuronList([skels,merges])
    skels, merges = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=min_nodes, query_dis=10, min_collin=min_collin, resample=None, smooth=None, split=False)
    skels = navis.NeuronList([skels,merges])
    write_navis_skels_tar(out_dir+"/Reconnected/"+name, skels)

    #translate strip and remove overlapping nodes
    tx,ty,tz=trans
    for bb in bound_boxs:
        skels = remove_translate_nodes(skels, trans=[0,0,0], bound_box=bb)
    skels = remove_translate_nodes(skels, trans=trans, bound_box=[0,0,0,0,0,0])
    write_navis_skels_tar(out_dir+"/Translated/"+name, skels)

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time results = Parallel(n_jobs=20)(delayed(postprocess_strip)(out_dir=out_dir, file=file, cl=cl, sc=sc, bound_boxs=bound_box, trans=trans) for file,bound_box,trans in zip(files,bound_boxs,translations))

In [ ]:
###Create lists of strips to reconnect
sc = load("/ACdata/Users/connorl/Models/scaler.joblib") #scaler file
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib") #model file

f1 = natsorted(glob.glob(out_dir + "*.gz"))[0:39]
f2 = natsorted(glob.glob(out_dir + "*.gz"))[39:78]
f3 = natsorted(glob.glob(out_dir + "*.gz"))[78:117]

y_combo1 = list(zip(f1, f1[1:]))[0::2] + list(zip(f2, f2[1:]))[0::2] + list(zip(f3, f3[1:]))[0::2]
y_combo2 = list(zip(f1, f1[1:]))[1::2] + list(zip(f2, f2[1:]))[1::2] + list(zip(f3, f3[1:]))[1::2]
z_combo = [(x) for x in zip(f1,f2)] + [[x] for x in zip(f2,f3)] 

In [ ]:
###Reconnect adjacent strips
def reconnect_strips(combo, cl, sc, edge_dis=[0,0,0]):
    print(combo)
    #combine neuronlist pairs and set strip names
    s1 = read_navis_neurons_tar(combo[0])
    s2 = read_navis_neurons_tar(combo[1])
    s1.set_neuron_attributes([combo[0]]*int(len(s1)), 'strip')
    s2.set_neuron_attributes([combo[1]]*int(len(s2)), 'strip')
    skels = navis.NeuronList([s1,s2])

    #find bounding box
    x,y,z = skels.nodes['x'],skels.nodes['y'],skels.nodes['z']
    x_dis,y_dis,z_dis = edge_dis
    x_range = list(range(*sorted(np.array([int(x.min()),int(x.max())]))))+[int(x.max())]
    y_range = list(range(*sorted(np.array([int(y.min()),int(y.max())]))))+[int(y.max())]
    z_range = list(range(*sorted(np.array([int(z.min()),int(z.max())]))))+[int(z.max())]
    bound_box = [x_range[x_dis],x_range[-1-x_dis],y_range[y_dis],y_range[-1-y_dis],z_range[z_dis],z_range[-1-z_dis]]

    try:
        #run reconnection
        non_merged, merged = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=20, query_dis=20, min_collin=0.6, resample=None, smooth=None, split=False, bound_box=bound_box)
        
        #replace strip files with merged skeletons removed
        for ind,strip in enumerate(list(set(non_merged.strip))):
            strip_sk = [i for i, value in enumerate(non_merged.strip) if value==strip]
            subset = non_merged[strip_sk]

            if ind==0:
                merged.set_neuron_attributes(['merged']*int(len(merged)), 'strip')
                subset = navis.NeuronList([subset,merged])
    
            #delete and replace strip
            os.remove(strip)
            write_navis_skels_tar(strip, subset)

    except:
        pass

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=20)(delayed(reconnect_strips)(combo=combo, cl=cl, sc=sc, edge_dis=[0,200,0]) for combo in y_combo1)

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=20)(delayed(reconnect_strips)(combo=combo, cl=cl, sc=sc, edge_dis=[0,200,0]) for combo in y_combo2)

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=20)(delayed(reconnect_strips)(combo=combo, cl=cl, sc=sc, edge_dis=[0,200,0]) for combo in z_combo)